# 🎯 CL-SDRG Phase 3: REINFORCE RL Training Engine

**Target:** Google Colab T4 GPU | **Estimated Time:** 20–30 min

This notebook:
1. Loads pre-processed training data from Phase 1
2. Pre-computes & caches embeddings from the frozen encoder
3. Trains the FGA using **REINFORCE policy gradient** with:
   - **Accuracy reward** R_acc = +1 (correct) / -1 (incorrect)
   - **Consistency reward** R_cons = 1 - L1(P_orig, P_perturbed)
   - **Combined** R_total = 0.6·R_acc + 0.4·R_cons
4. FP16 mixed precision + gradient accumulation (virtual batch=256)
5. Saves checkpoints and plots training curves

**⚠️ Prerequisites:** Run Notebook 01 first to generate `outputs/processed_data/fci_train.csv`

## 1. Setup

In [ ]:
!pip install -q torch transformers pandas tqdm matplotlib

In [ ]:
import os, sys, random, time, logging
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

# ── Logging ──
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)-8s %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(h)

# ── Seed ──
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Device: {device} ({torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU"})')

def fmt(n): return f'{n:,}'

# ── Config ──
ENCODER_NAME = 'intfloat/multilingual-e5-base'
EMBEDDING_DIM = 768
MAX_SEQ_LEN = 128
FGA_HIDDEN = 256
CLS_HIDDEN = 256
CLS_DROPOUT = 0.1
NUM_CLASSES = 3
LABEL2ID = {'TRUE': 0, 'FALSE': 1, 'MIXED': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Training
LAMBDA_ACC = 0.6
LAMBDA_CONS = 0.4
LR = 1e-4
WD = 0.01
BATCH_SIZE = 16
GRAD_ACCUM = 16
EPOCHS = 10
FP16 = True
BASELINE_DECAY = 0.99
CKPT_EVERY = 2

# Paths
PROC_DIR = Path('outputs/processed_data')
CKPT_DIR = Path('outputs/checkpoints')
FIG_DIR = Path('outputs/figures')
LOG_DIR = Path('outputs/logs')
for d in [CKPT_DIR, FIG_DIR, LOG_DIR]: d.mkdir(parents=True, exist_ok=True)

print(f'✅ Config loaded | vBatch={BATCH_SIZE*GRAD_ACCUM}')

## 2. Model Architecture

In [ ]:
class FrozenEncoder(nn.Module):
    def __init__(self, name=ENCODER_NAME):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.encoder = AutoModel.from_pretrained(name)
        for p in self.encoder.parameters(): p.requires_grad = False
        self.encoder.eval()
    @torch.no_grad()
    def encode(self, texts, dev, bs=64):
        prefixed = [f'query: {t}' for t in texts]
        tok = self.tokenizer(prefixed, max_length=MAX_SEQ_LEN, padding=True, truncation=True, return_tensors='pt').to(dev)
        out = self.encoder(**tok)
        m = tok['attention_mask'].unsqueeze(-1).float()
        return (out.last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1e-9)

class FeatureGatingAgent(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding_dim = EMBEDDING_DIM
        self.net = nn.Sequential(
            nn.Linear(EMBEDDING_DIM*3, FGA_HIDDEN), nn.ReLU(True),
            nn.Linear(FGA_HIDDEN, EMBEDDING_DIM*3), nn.Sigmoid())
    def forward(self, eq, es, et):
        g = self.net(torch.cat([eq,es,et], -1))
        return g.split(self.embedding_dim, -1)

class GatedFusion(nn.Module):
    def forward(self, eq,es,et,aq,as_,at):
        return aq*eq + as_*es + at*et

class VeracityClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.clf = nn.Sequential(
            nn.LayerNorm(EMBEDDING_DIM), nn.Linear(EMBEDDING_DIM, CLS_HIDDEN),
            nn.ReLU(True), nn.Dropout(CLS_DROPOUT), nn.Linear(CLS_HIDDEN, NUM_CLASSES))
    def forward(self, x): return self.clf(x)

print('✅ Architecture defined')

## 3. Load Data & Pre-compute Embeddings

In [ ]:
# ── Load training data ──
train_csv = PROC_DIR / 'fci_train.csv'
assert train_csv.exists(), f'Missing {train_csv}. Run Notebook 01 first!'

train_df = pd.read_csv(str(train_csv))
train_df['itemReviewed.author.name'] = train_df['itemReviewed.author.name'].fillna('Unknown')
train_df = train_df.dropna(subset=['claimReviewed', 'label_id'])
train_df['label_id'] = train_df['label_id'].astype(int)

logging.info(f'Training samples: {fmt(len(train_df))}')
for lid, cnt in train_df['label_id'].value_counts().sort_index().items():
    logging.info(f'  {ID2LABEL.get(lid,"?")}: {fmt(cnt)}')

In [ ]:
# ── Pre-compute embeddings ──
encoder = FrozenEncoder().to(device)

def encode_all(texts, desc, bs=64):
    embs = []
    for i in tqdm(range(0, len(texts), bs), desc=desc, unit='batch'):
        e = encoder.encode(texts[i:i+bs], device)
        embs.append(e.cpu())
    return torch.cat(embs)

claims = train_df['claimReviewed'].tolist()
speakers = train_df['itemReviewed.author.name'].tolist()
dates = train_df['datePublished'].astype(str).tolist()
labels = torch.tensor(train_df['label_id'].values, dtype=torch.long)

t0 = time.time()
claim_embs = encode_all(claims, 'Claims')

# Deduplicate speakers
unique_sp = list(set(speakers))
logging.info(f'Unique speakers: {fmt(len(unique_sp))}')
sp_map = {}
for i in tqdm(range(0, len(unique_sp), 64), desc='Speaker embeds'):
    batch = unique_sp[i:i+64]
    e = encoder.encode(batch, device).cpu()
    for s, emb in zip(batch, e): sp_map[s] = emb
speaker_embs = torch.stack([sp_map[s] for s in speakers])

date_embs = encode_all(dates, 'Dates')
logging.info(f'Embedding time: {time.time()-t0:.0f}s')
logging.info(f'Shapes: claims={claim_embs.shape}, speakers={speaker_embs.shape}, dates={date_embs.shape}')

# Free encoder
del encoder
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Dataset with Counterfactual Sampling

In [ ]:
class EmbCacheDS(Dataset):
    def __init__(self, ce, se, de, lbl, sp_names, sp_map):
        self.ce, self.se, self.de, self.lbl = ce, se, de, lbl
        self.sp_names = sp_names
        self.sp_map = sp_map
        self.all_sp = list(sp_map.keys())
    def __len__(self): return len(self.lbl)
    def __getitem__(self, i):
        return {'ce': self.ce[i], 'se': self.se[i], 'de': self.de[i],
                'lbl': self.lbl[i], 'sp': self.sp_names[i]}
    def cf_emb(self, orig_sp):
        cands = [s for s in self.all_sp if s != orig_sp]
        return self.sp_map[random.choice(cands)] if cands else self.sp_map[orig_sp]

ds = EmbCacheDS(claim_embs, speaker_embs, date_embs, labels, speakers, sp_map)

def collate(batch):
    return {
        'ce': torch.stack([b['ce'] for b in batch]),
        'se': torch.stack([b['se'] for b in batch]),
        'de': torch.stack([b['de'] for b in batch]),
        'lbl': torch.stack([b['lbl'] for b in batch]),
        'cf_se': torch.stack([ds.cf_emb(b['sp']) for b in batch]),
    }

dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate, drop_last=True)
logging.info(f'DataLoader: {len(dl)} batches')

## 5. REINFORCE Training Loop

In [ ]:
# ── Init trainable modules ──
fga = FeatureGatingAgent().to(device)
fusion = GatedFusion().to(device)
classifier = VeracityClassifier().to(device)

params = list(fga.parameters()) + list(classifier.parameters())
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
scaler = torch.amp.GradScaler('cuda', enabled=FP16 and device.type=='cuda')

tp = sum(p.numel() for p in params if p.requires_grad)
logging.info(f'Trainable params: {fmt(tp)}')

# ── Training ──
baseline = 0.0
history = []

print(f'\n{"="*70}')
print(f'  REINFORCE Training: {EPOCHS} epochs, vBatch={BATCH_SIZE*GRAD_ACCUM}')
print(f'  λ_acc={LAMBDA_ACC}, λ_cons={LAMBDA_CONS}, LR={LR}, FP16={FP16}')
print(f'{"="*70}\n')

for epoch in range(EPOCHS):
    fga.train(); classifier.train()
    losses, rewards, accs, cons = [], [], [], []
    correct = total = 0
    optimizer.zero_grad()

    pbar = tqdm(dl, desc=f'Epoch {epoch+1}/{EPOCHS}', unit='batch')
    for step, batch in enumerate(pbar):
        eq = batch['ce'].to(device)
        es = batch['se'].to(device)
        et = batch['de'].to(device)
        es_cf = batch['cf_se'].to(device)
        tgt = batch['lbl'].to(device)

        with torch.amp.autocast('cuda', enabled=FP16 and device.type=='cuda'):
            aq,as_,at = fga(eq, es, et)
            eg = fusion(eq,es,et,aq,as_,at)
            logits = classifier(eg)

            aq2,as2,at2 = fga(eq, es_cf, et)
            eg2 = fusion(eq,es_cf,et,aq2,as2,at2)
            logits_cf = classifier(eg2)

            probs = F.softmax(logits, -1)
            probs_cf = F.softmax(logits_cf, -1)
            preds = logits.argmax(-1)

            r_acc = (preds == tgt).float() * 2.0 - 1.0
            r_cons = 1.0 - torch.abs(probs - probs_cf).sum(-1)
            r_total = LAMBDA_ACC * r_acc + LAMBDA_CONS * r_cons

            log_p = F.log_softmax(logits, -1).gather(1, preds.unsqueeze(1)).squeeze(1)
            advantage = r_total - baseline
            policy_loss = -torch.mean(advantage.detach() * log_p)
            ce_loss = F.cross_entropy(logits, tgt)
            loss = (policy_loss + 0.5 * ce_loss) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step+1) % GRAD_ACCUM == 0 or (step+1) == len(dl):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()

        br = r_total.mean().item()
        baseline = BASELINE_DECAY * baseline + (1-BASELINE_DECAY) * br

        losses.append(loss.item() * GRAD_ACCUM)
        rewards.append(br)
        accs.append(r_acc.mean().item())
        cons.append(r_cons.mean().item())
        correct += (preds==tgt).sum().item()
        total += len(tgt)
        pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.4f}', rew=f'{np.mean(rewards[-50:]):.3f}', acc=f'{correct/total:.3f}')

    metrics = {
        'loss': float(np.mean(losses)), 'reward': float(np.mean(rewards)),
        'accuracy': correct/total, 'r_acc': float(np.mean(accs)),
        'r_cons': float(np.mean(cons)), 'baseline': baseline,
    }
    history.append(metrics)
    logging.info(f'Epoch {epoch+1} | Loss={metrics["loss"]:.4f} Rew={metrics["reward"]:.3f} '
                 f'Acc={metrics["accuracy"]:.3f} R_acc={metrics["r_acc"]:.3f} R_cons={metrics["r_cons"]:.3f}')

    if (epoch+1) % CKPT_EVERY == 0 or (epoch+1) == EPOCHS:
        ckpt = {'epoch': epoch+1, 'model_state_dict': {'fga': fga.state_dict(), 'classifier': classifier.state_dict()},
                'optimizer_state_dict': optimizer.state_dict(), 'metrics': metrics}
        torch.save(ckpt, str(CKPT_DIR / f'cl_sdrg_epoch_{epoch+1}.pt'))
        logging.info(f'💾 Checkpoint saved: epoch {epoch+1}')

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history)+1)
fig, axes = plt.subplots(2,2, figsize=(14,10))
fig.suptitle('CL-SDRG Training Progress', fontsize=16, fontweight='bold')

axes[0,0].plot(epochs, [h['loss'] for h in history], 'b-o', lw=2)
axes[0,0].set_title('Policy + CE Loss'); axes[0,0].set_xlabel('Epoch'); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(epochs, [h['reward'] for h in history], 'g-o', lw=2)
axes[0,1].set_title('Total Reward'); axes[0,1].set_xlabel('Epoch'); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(epochs, [h['r_acc'] for h in history], 'r-s', label='R_acc', lw=2)
axes[1,0].plot(epochs, [h['r_cons'] for h in history], 'm-^', label='R_cons', lw=2)
axes[1,0].set_title('Component Rewards'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(epochs, [h['accuracy'] for h in history], 'c-D', lw=2)
axes[1,1].set_title('Accuracy'); axes[1,1].set_xlabel('Epoch'); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIG_DIR/'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save history
pd.DataFrame(history).to_csv(str(LOG_DIR/'training_history.csv'), index=False)

print(f'\n{"="*70}')
print(f'  PHASE 3 COMPLETE')
print(f'  Best Accuracy: {max(h["accuracy"] for h in history):.4f}')
print(f'  Best Reward:   {max(h["reward"] for h in history):.4f}')
print(f'{"="*70}')
print('\n📋 Share the training curves and final metrics for Phase 4 planning.')